# CRANE+ V2 ビジョンピック — Colab の GPU で ACT を学習する

Mac（Docker）で集めて変換した LeRobotDataset を Google Drive 経由で受け取り、GPU で学習して Drive に保存します。

**事前準備（Mac 側）**
1. コンテナ内でデータセットを zip にして、マウントしているフォルダ（`/workspace`）に置く
   ```bash
   cd /home/ubuntu/dataset/lerobot_dataset && zip -r /workspace/crane_vision_pick.zip crane_vision_pick
   ```
2. Mac の Finder で `crane_vision_pick.zip` を Google Drive の `MyDrive/pai/` にアップロード

**メニュー →「ランタイム」→「ランタイムのタイプを変更」→ GPU（本課題では A100 を使用。T4 でも動く）** を選んでから実行してください。

In [ ]:
# GPU が割り当てられているか確認（CPU のままだと学習が数十倍遅い）
import torch, sys, platform
print(sys.version.split()[0], platform.platform())
if not torch.cuda.is_available():
    raise SystemExit("GPU がありません。メニューの「ランタイム → ランタイムのタイプを変更」で GPU（A100 / T4 など）を選び、"
                     "「ランタイム → すべてのセルを実行」をやり直してください。")
print("GPU:", torch.cuda.get_device_name(0))

## 1. 設定（ここだけ自分の環境に合わせて書き換える）

In [ ]:
REPO_URL    = ""   # GitHub に上げたら "https://github.com/<アカウント>/pai-crane-vision-pick.git"。空なら下の zip を使う
CODE_ZIP    = "/content/drive/MyDrive/pai/pai-crane-vision-pick.zip"   # REPO_URL が空のときに使うコードの zip
DRIVE_DIR   = "/content/drive/MyDrive/pai"                 # zip を置いたフォルダ
DATASET_ZIP = f"{DRIVE_DIR}/crane_vision_pick.zip"
TRAIN_CONFIG = "vision_train.yaml"                         # 画像なし比較は "state_only_train.yaml"
OUT_NAME    = "crane_vision_pick_act"                      # 画像なし比較は "crane_state_only_act"
STEPS       = 20000                                        # A100 で約 18 分（T4 なら 40〜60 分）。まず 200 で試してもよい

## 2. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. LeRobot（講義の pai_training と同じコミット）を入れる

Colab にはすでに PyTorch が入っているので、入れ直さないように `--no-deps` で LeRobot 本体だけ入れ、
必要な依存を追加します。今の Colab（PyTorch 2.11 / torchvision 0.26）では torchvision から動画の読み込み機能が
削除されているので、PyTorch の版に合った **TorchCodec** を入れて動画を読みます。

In [ ]:
# Colab の PyTorch はそのまま使い、LeRobot 本体と必要な依存だけ入れる
!pip install -q --no-deps "lerobot @ git+https://github.com/huggingface/lerobot.git@768e36660d1408c71118a2760f831c037fbfa17d"
!pip install -q "datasets==3.6.0" "huggingface_hub==0.32.3" "draccus==0.10.0" einops jsonlines termcolor deepdiff "av>=12" "imageio[ffmpeg]" safetensors pyyaml

# 動画の読み込みには TorchCodec が必要（torchvision 0.26 以降は VideoReader が削除されたため）
# 入っている PyTorch に対応する版を選ぶ（https://github.com/meta-pytorch/torchcodec の対応表）
import torch
mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
codec = {"2.7": "0.5", "2.8": "0.7", "2.9": "0.9", "2.10": "0.10", "2.11": "0.11"}.get(mm)
spec = f"torchcodec=={codec}.*" if codec else "torchcodec"
print("torch", torch.__version__, "→", spec)
!pip install -q "{spec}"

import lerobot, torchcodec
from torchcodec.decoders import VideoDecoder
print("torch", torch.__version__, "| torchcodec", torchcodec.__version__, "| cuda", torch.cuda.is_available())

## 4. プロジェクトのコードとデータセットを用意

In [ ]:
import os, subprocess
if not os.path.exists("/content/pai-crane-vision-pick"):
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "/content/pai-crane-vision-pick"], check=True)
    else:
        subprocess.run(["unzip", "-q", "-o", CODE_ZIP, "-d", "/content"], check=True)
os.makedirs("/content/dataset", exist_ok=True)
!unzip -q -o "{DATASET_ZIP}" -d /content/dataset
!ls /content/pai-crane-vision-pick/training/scripts /content/dataset/crane_vision_pick/meta
!head -c 600 /content/dataset/crane_vision_pick/meta/info.json

## 5. 学習（途中経過は 100 ステップごとに表示。モデルは Drive に保存）

In [ ]:
OUT = f"{DRIVE_DIR}/checkpoints/{OUT_NAME}"
!cd /content/pai-crane-vision-pick/training && python scripts/train_vision.py \
    --config configs/{TRAIN_CONFIG} --steps {STEPS} \
    --dataset-root /content/dataset/crane_vision_pick --output "{OUT}"

## 6. 学習データ上で予測と正解を比べる（第5回のデバッグ）

In [ ]:
!cd /content/pai-crane-vision-pick/training && python scripts/debug_policy_vision.py \
    --dataset-root /content/dataset/crane_vision_pick --repo-id crane_vision_pick \
    --policy-path "{OUT}" --episode-index 0 --device auto
from IPython.display import Image, display
display(Image(f"{OUT}/debug_ep0.png"))

## 7. Mac に持ち帰る

1. Google Drive で `MyDrive/pai/checkpoints/<OUT_NAME>.zip`（次のセルで作る zip）を Mac にダウンロード
2. ダブルクリックで展開し、できたフォルダ `<OUT_NAME>` を Mac の `~/physical/pai_work/checkpoints/` に置く
   （このフォルダはコンテナの中では `/workspace/checkpoints/` として見えるので、コピー作業は不要）
3. 中に `config.json` と `model.safetensors` があれば推論できます。検証 loss が一番良かったモデルは `best/` に入っています

In [ ]:
!cd "{DRIVE_DIR}/checkpoints" && zip -q -r "{OUT_NAME}.zip" "{OUT_NAME}" -x "*/step_*" && ls -lh "{OUT_NAME}.zip"